# 🏗️ Giorno 3 - Lab 1: Dal Project Canvas all'Architettura Agentica

Benvenuti al primo laboratorio del **Giorno 3**!

Questa mattina abbiamo parlato del **salto concettuale** che avviene quando si passa dalla teoria alla costruzione concreta:
- Il **Project Canvas** (compilato il Giorno 1) descrive *cosa* deve fare il sistema.
- L'**architettura agentica** descrive *come* costruirlo, con blocchi reali.
- **Langflow / LangChain** è il framework con cui costruiamo quei blocchi.

In questo notebook:
1. **Ripassiamo** il Project Canvas del caso guida (Assistente Acquisti Hardware).
2. **Traduciamo** ogni voce del canvas in un componente LangChain concreto.
3. **Esploriamo** i blocchi fondamentali: `ChatOllama`, `@tool`, `ChatPromptTemplate`, `AgentExecutor`.
4. **Definiamo** l'MVP: cosa mettiamo nel Giorno 3, cosa aggiungiamo nei giorni successivi.

---
### 🎯 Obiettivo
Alla fine di questo notebook avrai un **piano d'architettura chiaro** con la mappatura tra voci del canvas e classi/funzioni LangChain, pronto come riferimento per i notebook successivi.

> 💡 **LangChain & Langflow:** Langflow è costruito sopra LangChain. I componenti visivi di Langflow
> (Agent, ChatInput, Tool...) corrispondono esattamente a classi e funzioni LangChain.
> Lavorare con LangChain in Python ci dà visibilità completa su ciò che Langflow fa internamente.

## 1. Installazione e Verifica dell'Ambiente

Assicurati che Ollama sia avviato (`ollama serve`) e che `llama3.2` sia disponibile.

In [1]:
# Verifica dipendenze
import importlib, subprocess, sys

pacchetti = ["langchain", "langchain_ollama", "langchain_core"]
mancanti  = [p for p in pacchetti if importlib.util.find_spec(p) is None]
if mancanti:
    print(f"⚙️  Installazione: {mancanti}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + mancanti)
else:
    print("✅ Tutte le dipendenze sono già installate.")

# Verifica Ollama
import requests
try:
    resp   = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in resp.json().get("models", [])]
    ok     = any("llama3.2" in m for m in models)
    print("✅ Ollama raggiungibile — modelli:", models)
    print("✅ llama3.2 presente" if ok else "⚠️  llama3.2 NON trovato → ollama pull llama3.2")
except Exception as e:
    print(f"❌ Ollama non raggiungibile: {e}  →  avvia con: ollama serve")

✅ Tutte le dipendenze sono già installate.


c:\Users\A1618apulia\OneDrive - ITS Apulia Digital Maker\Desktop\GENERATIVEAI\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (6.0.0.post1)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


✅ Ollama raggiungibile — modelli: ['gpt-oss:20b', 'llama3.2:3b-instruct-q2_K', 'llama3.2:3b-instruct-q4_0', 'llama3.2:3b-instruct-q8_0', 'qwen2.5:latest', 'llama3.2:latest', 'nomic-embed-text:latest', 'qwen3.5:2b']
✅ llama3.2 presente


## 2. I Blocchi Fondamentali di LangChain

Prima di tradurre il canvas, conosciamo i mattoni di LangChain che useremo in tutti i lab successivi.

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage
try:
    from langchain.agents import AgentExecutor, create_tool_calling_agent
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

# ── Blocco 1: ChatOllama ─────────────────────────────────────────────────
# Corrisponde al componente "LLM / ChatModel" di Langflow
llm = ChatOllama(model="llama3.2", temperature=0)
print("✅ ChatOllama inizializzato — modello:", llm.model)

# Test rapido: una chiamata base
risposta = llm.invoke([HumanMessage(content="Rispondi con una sola parola: qual è la capitale dell'Italia?")])
print("   Risposta di test:", risposta.content)

✅ ChatOllama inizializzato — modello: llama3.2
   Risposta di test: Roma.


In [3]:
# ── Blocco 2: @tool decorator ─────────────────────────────────────────────
# Corrisponde al componente "Tool" di Langflow
# LangChain genera AUTOMATICAMENTE lo schema JSON dalla docstring e dai type hint

@tool
def esempio_tool(nome: str, valore: int) -> dict:
    """Restituisce un dizionario di esempio. Usare per dimostrare il funzionamento dei tool."""
    return {"nome": nome, "valore_doppio": valore * 2}

print("✅ Tool definito con @tool")
print("   Nome:        ", esempio_tool.name)
print("   Descrizione: ", esempio_tool.description)
print("   Schema args: ", esempio_tool.args)
print()
print("💡 LangChain ha generato lo schema JSON AUTOMATICAMENTE dalla docstring e dai type hint.")
print("   Nel Lab 2 scrivevamo lo schema a mano — ora non serve più.")

✅ Tool definito con @tool
   Nome:         esempio_tool
   Descrizione:  Restituisce un dizionario di esempio. Usare per dimostrare il funzionamento dei tool.
   Schema args:  {'nome': {'title': 'Nome', 'type': 'string'}, 'valore': {'title': 'Valore', 'type': 'integer'}}

💡 LangChain ha generato lo schema JSON AUTOMATICAMENTE dalla docstring e dai type hint.
   Nel Lab 2 scrivevamo lo schema a mano — ora non serve più.


In [4]:
# ── Blocco 3: ChatPromptTemplate ──────────────────────────────────────────
# Corrisponde al system prompt del componente "Agent" di Langflow

# Il template DEVE includere MessagesPlaceholder("agent_scratchpad")
# per lasciare spazio al ragionamento interno dell'agente (tool calls + risultati)
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Sei un assistente di esempio. Rispondi sempre in italiano."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✅ ChatPromptTemplate creato")
print("   Variabili di input:", prompt_template.input_variables)

✅ ChatPromptTemplate creato
   Variabili di input: ['agent_scratchpad', 'input']


In [5]:
# ── Blocco 4: create_tool_calling_agent + AgentExecutor ──────────────────
# Corrisponde al componente "Agent" di Langflow
# create_tool_calling_agent  → crea l'agente (il ragionatore)
# AgentExecutor              → gestisce il loop ReAct (reason → act → observe)

agent        = create_tool_calling_agent(llm, [esempio_tool], prompt_template)
executor     = AgentExecutor(agent=agent, tools=[esempio_tool], verbose=True)

# Test
risultato = executor.invoke({"input": "Usa il tool con nome=ciao e valore=21"})
print("\n📤 Output finale:", risultato["output"])



> Entering new AgentExecutor chain...

Invoking: `esempio_tool` with `{'nome': 'ciao', 'valore': 21}`


{'nome': 'ciao', 'valore_doppio': 42}Il tool "ciao" ha restituito il valore doppio del valore iniziale. Il valore doppio è stato calcolato come 21 x 2 = 42.

> Finished chain.

📤 Output finale: Il tool "ciao" ha restituito il valore doppio del valore iniziale. Il valore doppio è stato calcolato come 21 x 2 = 42.


## 3. Il Project Canvas del Caso Guida

Ora rileggiamo il canvas con gli occhi di un progettista LangChain.

In [6]:
import json

project_canvas = {
    "nome_progetto": "Assistente Aziendale Acquisto Hardware",
    "caso_duso": (
        "Un dipendente invia una richiesta di acquisto hardware. "
        "L'assistente AI verifica policy, budget e catalogo, poi decide:"
        " approvazione automatica, escalation al manager, o rifiuto."
    ),
    "input":  ["Richiesta in linguaggio naturale", "Badge ID dipendente"],
    "output": ["Approvazione automatica", "Escalation manager", "Rifiuto motivato"],
    "agenti": [{"nome": "Assistente Acquisti", "ruolo": "Orchestra i tool e formula la decisione"}],
    "tools": [
        {"nome": "get_employee_budget",    "descr": "Budget residuo e ruolo dipendente"},
        {"nome": "search_product_catalog", "descr": "Cerca prodotti nel catalogo"},
        {"nome": "request_purchase_approval", "descr": "Crea ticket escalation al manager"}
    ],
    "contesto": "Policy aziendale (nel system prompt)",
    "guardrail": ["Human approval per spese > soglia", "Rifiuto se prodotto non a catalogo"],
    "mvp_giorno3": "1 agente + 2 tool (budget + catalogo) + policy nel prompt"
}

print(json.dumps(project_canvas, indent=2, ensure_ascii=False))

{
  "nome_progetto": "Assistente Aziendale Acquisto Hardware",
  "caso_duso": "Un dipendente invia una richiesta di acquisto hardware. L'assistente AI verifica policy, budget e catalogo, poi decide: approvazione automatica, escalation al manager, o rifiuto.",
  "input": [
    "Richiesta in linguaggio naturale",
    "Badge ID dipendente"
  ],
  "output": [
    "Approvazione automatica",
    "Escalation manager",
    "Rifiuto motivato"
  ],
  "agenti": [
    {
      "nome": "Assistente Acquisti",
      "ruolo": "Orchestra i tool e formula la decisione"
    }
  ],
  "tools": [
    {
      "nome": "get_employee_budget",
      "descr": "Budget residuo e ruolo dipendente"
    },
    {
      "nome": "search_product_catalog",
      "descr": "Cerca prodotti nel catalogo"
    },
    {
      "nome": "request_purchase_approval",
      "descr": "Crea ticket escalation al manager"
    }
  ],
  "contesto": "Policy aziendale (nel system prompt)",
  "guardrail": [
    "Human approval per spese > soglia

## 4. Mappatura Canvas → Classi LangChain

| Voce del Canvas | Componente Langflow | Classe / Funzione LangChain |
|---|---|---|
| Richiesta dipendente | `Chat Input` | Stringa passata come `input` all'`AgentExecutor` |
| Agente principale | `Agent component` | `create_tool_calling_agent()` + `AgentExecutor` |
| LLM | LLM component | `ChatOllama(model="llama3.2")` |
| System prompt + policy | System prompt field | `ChatPromptTemplate` con `SystemMessage` |
| Tool: budget | `Custom Tool` | Funzione decorata con `@tool` |
| Tool: catalogo | `Custom Tool` | Funzione decorata con `@tool` |
| Tool: approvazione | `Custom Tool` | Funzione decorata con `@tool` |
| Guardrail / Human Approval | `Human Approval component` | Callback custom (Lab 3) |
| Agent-as-Tool | Agent in Tool Mode | `Tool.from_function()` che wrappa un secondo `AgentExecutor` (Lab 4) |
| Risposta finale | `Chat Output` | `risultato["output"]` dell'`AgentExecutor` |

In [7]:
# Schema visuale dell'architettura MVP
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║         ASSISTENTE ACQUISTI HARDWARE — ARCHITETTURA MVP (LangChain)     ║
╚══════════════════════════════════════════════════════════════════════════╝

  input (str)
       │
       ▼
  ┌─────────────────────────────────────────────────────────┐
  │          AgentExecutor (loop ReAct)                     │
  │                                                         │
  │  ┌──────────────────────────────────────────────────┐  │
  │  │  create_tool_calling_agent(llm, tools, prompt)   │  │
  │  │                                                  │  │
  │  │  ChatOllama(llama3.2)                            │  │
  │  │  ChatPromptTemplate(system + policy)             │  │
  │  └──────────────────────────────────────────────────┘  │
  │                                                         │
  │  Tools:                                                 │
  │   @tool  get_employee_budget(employee_id)               │
  │   @tool  search_product_catalog(query)                  │
  └──────────────────────────────┬──────────────────────────┘
                                 │
                                 ▼
                        risultato["output"]  →  Risposta al dipendente

  DATI: data/employees.json  |  data/products.json  |  data/policy_hardware.md
  MVP:  1 AgentExecutor  +  2 @tool  (sola lettura)
""")


╔══════════════════════════════════════════════════════════════════════════╗
║         ASSISTENTE ACQUISTI HARDWARE — ARCHITETTURA MVP (LangChain)     ║
╚══════════════════════════════════════════════════════════════════════════╝

  input (str)
       │
       ▼
  ┌─────────────────────────────────────────────────────────┐
  │          AgentExecutor (loop ReAct)                     │
  │                                                         │
  │  ┌──────────────────────────────────────────────────┐  │
  │  │  create_tool_calling_agent(llm, tools, prompt)   │  │
  │  │                                                  │  │
  │  │  ChatOllama(llama3.2)                            │  │
  │  │  ChatPromptTemplate(system + policy)             │  │
  │  └──────────────────────────────────────────────────┘  │
  │                                                         │
  │  Tools:                                                 │
  │   @tool  get_employee_budget(employee_id)               

## 5. Evoluzione dell'Architettura

In [8]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                EVOLUZIONE DELL'ARCHITETTURA                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  GIORNO 3 — MVP (Lab 2)                                          ║
║    ChatOllama + 2 @tool + AgentExecutor                          ║
║                                                                  ║
║  GIORNO 3 — Iterazione (Lab 3 + Lab 4)                          ║
║    + Human Approval (callback / interrupt)                       ║
║    + Agent-as-Tool (Tool.from_function che wrappa AgentExecutor) ║
║                                                                  ║
║  GIORNO 4                                                        ║
║    LangGraph: StateGraph, nodi, conditional_edges, interrupt()   ║
║                                                                  ║
║  GIORNO 5                                                        ║
║    + ConversationBufferMemory / RunnableWithMessageHistory        ║
║    + RAG con FAISS / Chroma + langchain-community retrievers     ║
╚══════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════╗
║                EVOLUZIONE DELL'ARCHITETTURA                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  GIORNO 3 — MVP (Lab 2)                                          ║
║    ChatOllama + 2 @tool + AgentExecutor                          ║
║                                                                  ║
║  GIORNO 3 — Iterazione (Lab 3 + Lab 4)                          ║
║    + Human Approval (callback / interrupt)                       ║
║    + Agent-as-Tool (Tool.from_function che wrappa AgentExecutor) ║
║                                                                  ║
║  GIORNO 4                                                        ║
║    LangGraph: StateGraph, nodi, conditional_edges, interrupt()   ║
║                                                                  ║
║  GIORNO 5                       

## 6. 💡 Esercizio: Esplora lo Schema di un Tool

Definisci un tool con `@tool` per il tuo caso d'uso (anche inventato) e osserva lo schema generato automaticamente da LangChain.

In [ ]:
# TODO: Definisci il tuo primo @tool
# Suggerimento: pensa a un tool che legge dati dal tuo dominio

@tool
def mio_primo_tool(parametro: str) -> dict:
    """TODO: scrivi una descrizione chiara di QUANDO questo tool deve essere invocato."""
    return {"risultato": f"Output per: {parametro}"}

print("Nome:       ", mio_primo_tool.name)
print("Descrizione:", mio_primo_tool.description)
print("Args schema:")
import json
print(json.dumps(mio_primo_tool.args_schema.model_json_schema(), indent=2, ensure_ascii=False))

Nome:        mio_primo_tool
Descrizione: TODO: scrivi una descrizione chiara di QUANDO questo tool deve essere invocato.
Args schema:
{
  "description": "TODO: scrivi una descrizione chiara di QUANDO questo tool deve essere invocato.",
  "properties": {
    "parametro": {
      "title": "Parametro",
      "type": "string"
    }
  },
  "required": [
    "parametro"
  ],
  "title": "mio_primo_tool",
  "type": "object"
}


: 

---
### ✅ Recap Lab 1

- Abbiamo installato e verificato **LangChain + langchain-ollama**.
- Abbiamo esplorato i 4 blocchi fondamentali: `ChatOllama`, `@tool`, `ChatPromptTemplate`, `AgentExecutor`.
- Abbiamo mappato ogni voce del Project Canvas sulla corrispondente classe/funzione LangChain.
- Abbiamo visto che `@tool` **genera automaticamente lo schema JSON** da docstring e type hints.

➡️ **Prossimo notebook:** `02_Agente_con_Tools_Ollama.ipynb` — costruiamo l'MVP completo e lo testiamo su 3 scenari.